## Stage 0 — Setup

Imports and header so worth athletics doesn't pick it up as a bot.

In [1]:
import requests
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup
from io import StringIO

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

## Stage 1 — Scrape a single race
Here we scrap the races from the results page and get this information:
athlete, country, time, record flag 
- reaction time
- wind reading
- date, venue, meet name

Handles non-numeric results (DNF/DQ/DNS) by keeping the row with a NaN time,
since a DNF is still a real data point (something went wrong that race).

In [2]:
def clean_mark(mark):
    """Split '9.79 PB' into (9.79, 'PB'), or '9.91' into (9.91, None).
    Non-numeric marks (DNF/DQ/DNS) get NaN time and the code as the flag."""
    if pd.isna(mark):
        return np.nan, None
    parts = str(mark).strip().split() # whitespace
    try:
        time = float(parts[0]) 
        flag = parts[1] if len(parts) > 1 else None
    except ValueError:
        time = np.nan
        flag = parts[0]  # e.g. 'DNF', 'DQ', 'DNS'
    return time, flag


def scrape_race(url):
    resp = requests.get(url, headers=HEADERS, timeout=15) # timeout in seconds
    resp.raise_for_status() # raises an error if the request failed

    tables = pd.read_html(StringIO(resp.text)) # parse all tables from the HTML
    results_table = max(tables, key=len) # get largest tble
    
    # Normalize column names across inconsistent page formats
    results_table.columns = [str(c).strip() for c in results_table.columns]
    rename_map = {
        "Mark": "MARK",
        "Athlete": "ATHLETE",
        "Unnamed: 2": "COUNTRY",
        }
    results_table = results_table.rename(columns=rename_map)

    results_table[["time", "record_flag"]] = results_table["MARK"].apply(
        lambda m: pd.Series(clean_mark(m))
    )

    soup = BeautifulSoup(resp.text, "html.parser") # parse the HTML with BeautifulSoup

    # wind
    heading = soup.find(string=lambda s: s and "Wind" in s) # find wind 
    wind = None
    if heading:
        match = re.search(r"Wind\s*([+-]?\d+\.\d+)", heading) # regex for finding wind speed
        if match:
            wind = float(match.group(1))

    # date — from EventDate meta tag 
    date = None
    for meta in soup.find_all("meta"):
        if meta.get("name") == "EventDate" or meta.get("property") == "EventDate":
            date = meta.get("content")
            break

    # venue + meet name
    desc_tag = soup.find("meta", attrs={"property": "og:description"})
    venue = None
    if desc_tag and desc_tag.get("content"):
        venue_match = re.search(r"in\s+([A-Za-z\s]+)$", desc_tag["content"].strip())
        if venue_match:
            venue = venue_match.group(1).strip()

    title_tag = soup.find("meta", attrs={"property": "og:title"})
    meet_name = title_tag["content"].strip() if title_tag and title_tag.get("content") else None

    results_table["wind"] = wind
    results_table["date"] = date
    results_table["venue"] = venue
    results_table["meet_name"] = meet_name
    results_table["source_url"] = url

    return results_table[["ATHLETE", "COUNTRY", "time", "record_flag", "Reaction Time",
                            "wind", "date", "venue", "meet_name", "source_url"]]

## Stage 2 — Test across multiple races

One race working isn't enough proof — different meets can have messier data
(DNF/DQ results) or slightly different page structures. Here we run the
scraper across 4 different races to find where it breaks.

In [3]:
urls = [
    "https://worldathletics.org/results/olympic-games/2024/the-xxxiii-olympic-games-7153115/men/100-metres/final/result",  # Paris 2024
    "https://worldathletics.org/results/world-athletics-championships/2023/world-athletics-championships-budapest-2023-7138987/men/100-metres/final/result",  # Budapest 2023
    "https://worldathletics.org/results/olympic-games/2021/the-xxxii-olympic-games-7132391/men/100-metres/final/result",  # Tokyo 2020(21) - has a DNF
    "https://worldathletics.org/competitions/world-athletics-championships/world-athletics-championships-oregon-2022-7137279/results/men/100-metres/final/result",  # Oregon 2022 - different URL structure
]

all_races = []
for u in urls:
    try:
        race = scrape_race(u)
        all_races.append(race)
        print(f"OK ({len(race)} rows): {u}")
    except Exception as e:
        print(f"FAILED: {u}")
        print(f"   error: {e}")

full_dataset = pd.concat(all_races, ignore_index=True)
full_dataset

OK (8 rows): https://worldathletics.org/results/olympic-games/2024/the-xxxiii-olympic-games-7153115/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/results/world-athletics-championships/2023/world-athletics-championships-budapest-2023-7138987/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/results/olympic-games/2021/the-xxxii-olympic-games-7132391/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/competitions/world-athletics-championships/world-athletics-championships-oregon-2022-7137279/results/men/100-metres/final/result


C:\Users\rotem\AppData\Local\Temp\ipykernel_16424\656437860.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full_dataset = pd.concat(all_races, ignore_index=True)


,ATHLETE,COUNTRY,time,record_flag,Reaction Time,wind,date,venue,meet_name,source_url
0,Noah LYLES,USA,9.79,PB,0.178,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
1,Kishane THOMPSON,JAM,9.79,NaN,0.176,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
2,Fred KERLEY,USA,9.81,SB,0.108,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
3,Akani SIMBINE,RSA,9.82,NR,0.149,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
4,Lamont Marcell JACOBS,ITA,9.85,SB,0.114,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
5,Letsile TEBOGO,BOT,9.86,NR,0.178,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
6,Kenneth BEDNAREK,USA,9.88,NaN,0.163,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
7,Oblique SEVILLE,JAM,9.91,NaN,0.171,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
8,Noah LYLES,USA,9.83,=WL,0.145,0.0,19/08/2023 00:00:00,Budapest,World Athletics | 100 Metres Result | World A...,https://worldathletics.org/results/world-athle...
9,Letsile TEBOGO,BOT,9.88,NR,0.129,0.0,19/08/2023 00:00:00,Budapest,World Athletics | 100 Metres Result | World A...,https://worldathletics.org/results/world-athle...


## Stage 3 — Handle inconsistent page formats

Running the scraper across multiple races (Stage 2) revealed that not every
World Athletics results page is structured the same way. Two real bugs
turned up:

1. Tokyo 2020(21) had a DNF result — 'MARK' contained the text "DNF" instead
   of a time, which crashed the original clean_mark() function. Fixed by
   catching non-numeric marks and storing them as a record_flag instead of
   a time.

2. Oregon 2022 uses a different URL structure (/competitions/.../results/...
   instead of /results/...), and its results table uses different column
   names ('Mark' instead of 'MARK',

In [4]:
resp = requests.get(
    "https://worldathletics.org/competitions/world-athletics-championships/world-athletics-championships-oregon-2022-7137279/results/men/100-metres/final/result",
    headers=HEADERS, timeout=15
)
tables = pd.read_html(StringIO(resp.text))
print(f"Found {len(tables)} tables")

results_table = max(tables, key=len)
print(results_table.columns.tolist())
results_table

Found 1 tables
['Pos', 'Bib', 'Unnamed: 2', 'Athlete', 'Mark', 'Reaction Time']


,Pos,Bib,Unnamed: 2,Athlete,Mark,Reaction Time
0,1,2389,USA,Fred KERLEY,9.86,0.119
1,2,2360,USA,Marvin BRACY,9.88,0.118
2,3,2363,USA,Trayvon BROMELL,9.88,0.110
3,4,2039,JAM,Oblique SEVILLE,9.97,0.154
4,5,2266,RSA,Akani SIMBINE,10.01,0.129
5,6,2368,USA,Christian COLEMAN,10.01,0.104
6,7,2068,JPN,Abdul Hakim SANI BROWN,10.06,0.147
7,8,1639,CAN,Aaron BROWN,10.07,0.155


## Stage 4 — Build the real dataset

Collect a batch of race URLs (Olympic finals, World Champs finals, a few
Diamond League races), run scrape_race() across all of them, and save
the combined result to a CSV so we're not re-scraping every time we
restart the kernel.

Known gap: pages using the `/competitions/.../results/...` URL structure
(e.g. Oregon 2022) don't return wind/date/venue — only athlete/time/reaction
time. Accepted as a limitation for now rather than fixed.

In [5]:
urls = [
    "https://worldathletics.org/results/olympic-games/2024/the-xxxiii-olympic-games-7153115/men/100-metres/final/result",  # Paris 2024
    "https://worldathletics.org/results/world-athletics-championships/2023/world-athletics-championships-budapest-2023-7138987/men/100-metres/final/result",  # Budapest 2023
    "https://worldathletics.org/results/olympic-games/2021/the-xxxii-olympic-games-7132391/men/100-metres/final/result",  # Tokyo 2020(21)
    "https://worldathletics.org/competitions/world-athletics-championships/world-athletics-championships-oregon-2022-7137279/results/men/100-metres/final/result",  # Oregon 2022 - known gap: no wind/date/venue
    "https://worldathletics.org/results/world-athletics-championships/2019/iaaf-world-athletics-championships-doha-2019-7125365/men/100-metres/final/result",  # Doha 2019
    "https://worldathletics.org/results/world-athletics-championships/2017/iaaf-world-championships-london-2017-7093740/men/100-metres/final/result", 
    "https://worldathletics.org/results/iaaf-world-championships-in-athletics/2015/15th-iaaf-world-championships-7078726/men/100-metres/final/result",  # Beijing 2015 - Bolt beats Gatlin by 0.01s
    "https://worldathletics.org/results/world-athletics-championships/2013/14th-iaaf-world-championships-7003368/men/100-metres/final/result",  # Moscow 2013
    "https://worldathletics.org/results/olympic-games/2012/the-xxx-olympic-games-6999193/men/100-metres/final/result",  # London 2012 - Bolt's 9.63 Olympic Record
    "https://worldathletics.org/results/world-athletics-championships/2011/13th-iaaf-world-championships-in-athletics-7003367/men/100-metres/final/result",  # Daegu 2011 - Bolt DQ'd for false start# London 2017 - Bolt's last individual race
]

all_races = []
for u in urls:
    try:
        race = scrape_race(u)
        all_races.append(race)
        print(f"OK ({len(race)} rows): {u}")
    except Exception as e:
        print(f"FAILED: {u}")
        print(f"   error: {e}")

full_dataset = pd.concat(all_races, ignore_index=True)
full_dataset

OK (8 rows): https://worldathletics.org/results/olympic-games/2024/the-xxxiii-olympic-games-7153115/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/results/world-athletics-championships/2023/world-athletics-championships-budapest-2023-7138987/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/results/olympic-games/2021/the-xxxii-olympic-games-7132391/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/competitions/world-athletics-championships/world-athletics-championships-oregon-2022-7137279/results/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/results/world-athletics-championships/2019/iaaf-world-athletics-championships-doha-2019-7125365/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/results/world-athletics-championships/2017/iaaf-world-championships-london-2017-7093740/men/100-metres/final/result
OK (9 rows): https://worldathletics.org/results/iaaf-world-championships-in-athletics/2015/15

C:\Users\rotem\AppData\Local\Temp\ipykernel_16424\2088878736.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full_dataset = pd.concat(all_races, ignore_index=True)


,ATHLETE,COUNTRY,time,record_flag,Reaction Time,wind,date,venue,meet_name,source_url
0,Noah LYLES,USA,9.79,PB,0.178,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
1,Kishane THOMPSON,JAM,9.79,NaN,0.176,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
2,Fred KERLEY,USA,9.81,SB,0.108,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
3,Akani SIMBINE,RSA,9.82,NR,0.149,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
4,Lamont Marcell JACOBS,ITA,9.85,SB,0.114,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
...,...,...,...,...,...,...,...,...,...,...
76,Christophe LEMAITRE,FRA,10.19,NaN,0.179,-1.4,27/08/2011 00:00:00,None,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
77,Daniel Bailey,ANT,10.26,NaN,0.165,-1.4,27/08/2011 00:00:00,None,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
78,Jimmy VICAUT,FRA,10.27,NaN,0.162,-1.4,27/08/2011 00:00:00,None,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
79,Nesta CARTER,JAM,10.95,NaN,0.154,-1.4,27/08/2011 00:00:00,None,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...


## Stage 5 — Feature engineering

Load the saved dataset and start turning race-level rows into athlete-level
features. First step: clean up data types (dates, names) so grouping by
athlete actually works correctly — then build the simplest useful feature
(recency-weighted average time) before anything fancier.

In [6]:
df = pd.read_csv("100m_races_dataset.csv")

# Normalize athlete names so the same person isn't split into two
# groups due to inconsistent capitalization (e.g. "Noah LYLES" vs "Noah Lyles")
df["ATHLETE"] = df["ATHLETE"].str.strip().str.title()

# Convert date column from text to an actual datetime, so we can sort
# chronologically and compute "most recent races"
df["date"] = pd.to_datetime(df["date"], format="%d/%m/%Y %H:%M:%S", errors="coerce")

print(df["ATHLETE"].nunique(), "unique athletes")
print(df["date"].min(), "to", df["date"].max())
df.sort_values("date").head(10)

40 unique athletes
2011-08-27 00:00:00 to 2024-08-01 00:00:00


,ATHLETE,COUNTRY,time,record_flag,Reaction Time,wind,date,venue,meet_name,source_url
80,Usain Bolt,JAM,NaN,DQ,-0.104,-1.4,2011-08-27,NaN,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
73,Yohan Blake,JAM,9.92,SB,0.174,-1.4,2011-08-27,NaN,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
74,Walter Dix,USA,10.08,NaN,0.175,-1.4,2011-08-27,NaN,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
75,Kim Collins,SKN,10.09,NaN,0.155,-1.4,2011-08-27,NaN,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
79,Nesta Carter,JAM,10.95,NaN,0.154,-1.4,2011-08-27,NaN,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
77,Daniel Bailey,ANT,10.26,NaN,0.165,-1.4,2011-08-27,NaN,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
78,Jimmy Vicaut,FRA,10.27,NaN,0.162,-1.4,2011-08-27,NaN,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
76,Christophe Lemaitre,FRA,10.19,NaN,0.179,-1.4,2011-08-27,NaN,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
72,Tyson Gay,USA,NaN,DQ,0.145,1.5,2012-07-27,NaN,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
71,Asafa Powell,JAM,11.99,NaN,0.155,1.5,2012-07-27,NaN,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...


In [7]:
race_counts = df.groupby("ATHLETE").size().sort_values(ascending=False)
race_counts.head(15)

ATHLETE
Akani Simbine             5
Justin Gatlin             5
Usain Bolt                5
Christian Coleman         4
Yohan Blake               4
Jimmy Vicaut              3
Bingtian Su               3
Andre De Grasse           3
Zharnel Hughes            3
Oblique Seville           3
Fred Kerley               3
Aaron Brown               2
Christophe Lemaitre       2
Asafa Powell              2
Abdul Hakim Sani Brown    2
dtype: int64

### Feature 1 — Per-athlete average time and consistency

For athletes with multiple races, compute their average time and the
standard deviation of their times (how consistent they are). Athletes
with only 1 race get a mean but no meaningful std — flagged separately
rather than treated as "perfectly consistent."

In [8]:
athlete_stats = df.groupby("ATHLETE")["time"].agg(
    races="count",
    avg_time="mean",
    consistency="std"
).sort_values("avg_time")

athlete_stats

,races,avg_time,consistency
ATHLETE,,,
Usain Bolt,4,9.785000,0.131022
Kishane Thompson,1,9.790000,NaN
Noah Lyles,2,9.810000,0.028284
Lamont Marcell Jacobs,2,9.825000,0.035355
Fred Kerley,3,9.836667,0.025166
Justin Gatlin,5,9.850000,0.056125
Letsile Tebogo,2,9.870000,0.014142
Kenneth Bednarek,1,9.880000,NaN
Ryan Bailey,1,9.880000,NaN


### Feature 2 — Recency-weighted average time

Instead of a flat average, weight each race by how recent it is. Older
races (like a one-off injury years ago) count less than a runner's
current form. Uses exponential decay: a race's weight halves every
~2 years, so recent performances dominate the average without
completely ignoring history.

In [9]:
import numpy as np

reference_date = df["date"].max()  # most recent race in the whole dataset
half_life_days = 365 * 2           # weight halves every ~2 years

def recency_weighted_avg(group):
    valid = group.dropna(subset=["time", "date"])  # <-- now requires both
    if len(valid) == 0:
        return pd.Series({"weighted_avg_time": np.nan, "races_used": 0})
    
    days_ago = (reference_date - valid["date"]).dt.days
    weights = 0.5 ** (days_ago / half_life_days)
    
    weighted_avg = (valid["time"] * weights).sum() / weights.sum()
    return pd.Series({"weighted_avg_time": weighted_avg, "races_used": len(valid)})

recency_stats = df.groupby("ATHLETE").apply(recency_weighted_avg, include_groups=False)
recency_stats.sort_values("weighted_avg_time")

,weighted_avg_time,races_used
ATHLETE,,
Kishane Thompson,9.790000,1.0
Noah Lyles,9.806725,2.0
Fred Kerley,9.817820,2.0
Lamont Marcell Jacobs,9.836967,2.0
Usain Bolt,9.855612,4.0
Akani Simbine,9.866752,4.0
Letsile Tebogo,9.868363,2.0
Justin Gatlin,9.879280,5.0
Kenneth Bednarek,9.880000,1.0


## Stage 6 — Baseline model

The simplest possible prediction: given a list of athletes, rank them by
their recency-weighted average time. Fastest weighted average = predicted
winner. No machine learning yet — this is the benchmark every future
model (Monte Carlo, Plackett-Luce, XGBoost) has to actually beat to be
worth the added complexity.

In [10]:
def predict_race(athlete_names, stats_table=recency_stats):
    """Given a list of athlete names, rank them by recency-weighted average time.
    Lower time = predicted higher finish."""
    field = stats_table.loc[stats_table.index.intersection(athlete_names)]
    field = field.dropna(subset=["weighted_avg_time"]).sort_values("weighted_avg_time")
    
    missing = set(athlete_names) - set(field.index)
    if missing:
        print(f"Warning: no usable data for {missing} — excluded from prediction")
    
    return field

# Test it on a real 8-person field — e.g. reconstruct the actual Paris 2024 final
paris_field = ["Noah Lyles", "Kishane Thompson", "Fred Kerley", "Akani Simbine",
               "Lamont Marcell Jacobs", "Letsile Tebogo", "Kenneth Bednarek", "Oblique Seville"]

predict_race(paris_field)

,weighted_avg_time,races_used
ATHLETE,,
Kishane Thompson,9.790000,1.0
Noah Lyles,9.806725,2.0
Fred Kerley,9.817820,2.0
Lamont Marcell Jacobs,9.836967,2.0
Akani Simbine,9.866752,4.0
Letsile Tebogo,9.868363,2.0
Kenneth Bednarek,9.880000,1.0
Oblique Seville,9.897456,2.0


In [11]:
budapest_field = ["Noah Lyles", "Letsile Tebogo", "Zharnel Hughes", "Oblique Seville",
                   "Christian Coleman", "Abdul Hakim Sani Brown", "Ferdinand Omanyala", "Ryiem Forde"]

predict_race(budapest_field)

,weighted_avg_time,races_used
ATHLETE,,
Noah Lyles,9.806725,2.0
Letsile Tebogo,9.868363,2.0
Christian Coleman,9.891781,3.0
Oblique Seville,9.897456,2.0
Zharnel Hughes,9.910875,2.0
Abdul Hakim Sani Brown,10.040000,1.0
Ferdinand Omanyala,10.070000,1.0
Ryiem Forde,10.080000,1.0


In [12]:
beijing_field = ["Usain Bolt", "Justin Gatlin", "Christophe Lemaitre",
                  "Trayvon Bromell", "Andre De Grasse", "Bingtian Su"]

predict_race(beijing_field)


,weighted_avg_time,races_used
ATHLETE,,
Usain Bolt,9.855612,4.0
Justin Gatlin,9.879280,5.0
Andre De Grasse,9.895501,3.0
Trayvon Bromell,9.920000,1.0
Bingtian Su,10.040195,3.0
Christophe Lemaitre,10.103773,2.0


## Stage 7 — Monte Carlo simulation

Instead of ranking athletes by a single average time, model each
athlete's performance as a distribution: a mean (their weighted average)
and a spread (their consistency/std). Simulate the race thousands of
times, drawing a random time from each athlete's distribution each run,
and see who "wins" most often. This produces win probabilities instead
of a single ranked guess — a more honest way to describe a sport this close.

In [13]:
def simulate_race(athlete_names, stats_table, n_simulations=10000):
    field = stats_table.loc[stats_table.index.intersection(athlete_names)].copy()
    field = field.dropna(subset=["weighted_avg_time"])
    
    # For athletes with only 1 race, we have no real std — fall back to
    # the average std of everyone else in the field as a rough estimate
    fallback_std = field["consistency"].mean() if "consistency" in field else 0.05
    
    missing = set(athlete_names) - set(field.index)
    if missing:
        print(f"Warning: no usable data for {missing} — excluded from simulation")
    
    wins = {name: 0 for name in field.index}
    
    for _ in range(n_simulations):
        simulated_times = {}
        for name, row in field.iterrows():
            std = row["consistency"] if pd.notna(row.get("consistency")) else fallback_std
            simulated_times[name] = np.random.normal(row["weighted_avg_time"], std)
        winner = min(simulated_times, key=simulated_times.get)
        wins[winner] += 1
    
    win_probs = pd.Series(wins) / n_simulations
    return win_probs.sort_values(ascending=False)

In [14]:
recency_stats = recency_stats.join(athlete_stats["consistency"])
recency_stats.head()

,weighted_avg_time,races_used,consistency
ATHLETE,,,
Aaron Brown,10.080000,1.0,0.007071
Abdul Hakim Sani Brown,10.040000,1.0,0.014142
Akani Simbine,9.866752,4.0,0.078102
Andre De Grasse,9.895501,3.0,0.015275
Asafa Powell,10.510371,2.0,1.407142


In [15]:
simulate_race(paris_field, recency_stats)

Kishane Thompson         0.4678
Noah Lyles               0.2144
Akani Simbine            0.1323
Fred Kerley              0.1040
Lamont Marcell Jacobs    0.0667
Oblique Seville          0.0083
Kenneth Bednarek         0.0064
Letsile Tebogo           0.0001
dtype: float64

### Fixing data leakage — cutoff-based stats

The earlier simulation accidentally used Kishane Thompson's actual Paris
2024 result as his "prior form" when predicting Paris 2024 — this is
data leakage, since we're using the answer to predict the answer.

Fix: build a function that computes recency-weighted stats using only
races before a given cutoff date. When backtesting a race, the cutoff
is that race's date — so every athlete's stats reflect only what was
knowable beforehand.

In [16]:
def compute_stats_before(cutoff_date, df=df):
    """Recompute recency-weighted stats using only races strictly before cutoff_date.
    Prevents leaking the target race's own result into an athlete's 'known form'."""
    history = df[df["date"] < cutoff_date]
    
    def recency_weighted_avg(group):
        valid = group.dropna(subset=["time", "date"])
        if len(valid) == 0:
            return pd.Series({"weighted_avg_time": np.nan, "races_used": 0, "consistency": np.nan})
        
        days_ago = (cutoff_date - valid["date"]).dt.days
        weights = 0.5 ** (days_ago / half_life_days)
        weighted_avg = (valid["time"] * weights).sum() / weights.sum()
        consistency = valid["time"].std()  # simple std of prior races
        
        return pd.Series({
            "weighted_avg_time": weighted_avg,
            "races_used": len(valid),
            "consistency": consistency
        })
    
    return history.groupby("ATHLETE").apply(recency_weighted_avg, include_groups=False)

# Rebuild stats using only races BEFORE Paris 2024
paris_date = pd.Timestamp("2024-08-01")
stats_before_paris = compute_stats_before(paris_date)

stats_before_paris.loc[stats_before_paris.index.intersection(paris_field)]

,weighted_avg_time,races_used,consistency
ATHLETE,,,
Akani Simbine,9.941283,3.0,0.046188
Fred Kerley,9.840000,1.0,NaN
Lamont Marcell Jacobs,9.800000,1.0,NaN
Letsile Tebogo,9.880000,1.0,NaN
Noah Lyles,9.830000,1.0,NaN
Oblique Seville,9.880000,1.0,NaN


In [17]:
simulate_race(paris_field, stats_before_paris)

Lamont Marcell Jacobs    0.5263
Noah Lyles               0.2274
Fred Kerley              0.1673
Oblique Seville          0.0399
Letsile Tebogo           0.0375
Akani Simbine            0.0016
dtype: float64

## Stage 8 — Automated URL discovery

Instead of manually finding race URLs one at a time, scrape World
Athletics' own meeting-index pages to discover URLs automatically.
This is a two-hop process: scrape an index page for meeting links,
then visit each meeting page to find the men's 100m event link(s),
then feed those into scrape_race(). Enables true large-scale collection
(hundreds of races) instead of a hand-picked handful.

In [18]:
resp = requests.get("https://worldathletics.org/results/diamond-league-meetings", headers=HEADERS, timeout=15)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

links = soup.find_all("a", href=True)
# Only keep links that actually START with /results/diamond-league-meetings/
# (filters out the Facebook/Twitter share junk that just contained the string)
meeting_links = [
    a["href"] for a in links
    if a["href"].startswith("/results/diamond-league-meetings/")
]

# Convert relative URLs to full URLs
meeting_links = ["https://worldathletics.org" + link for link in meeting_links]

# Remove duplicates while preserving order
meeting_links = list(dict.fromkeys(meeting_links))

print(f"Found {len(meeting_links)} clean meeting links")
for link in meeting_links[:10]:
    print(link)

Found 30 clean meeting links
https://worldathletics.org/results/diamond-league-meetings/2021/weltklasse-zurich-7058
https://worldathletics.org/results/diamond-league-meetings/2021/allianz-memorial-van-damme-7056
https://worldathletics.org/results/diamond-league-meetings/2021/meeting-de-paris-7055
https://worldathletics.org/results/diamond-league-meetings/2021/athletissima-lausanne-7054
https://worldathletics.org/results/diamond-league-meetings/2021/prefontaine-classic-7044
https://worldathletics.org/results/diamond-league-meetings/2021/muller-british-gp-7051
https://worldathletics.org/results/diamond-league-meetings/2021/herculis-7052
https://worldathletics.org/results/diamond-league-meetings/2021/bauhaus-galan-7050
https://worldathletics.org/results/diamond-league-meetings/2021/oslo-bislett-games-7049
https://worldathletics.org/results/diamond-league-meetings/2021/golden-gala-pietro-mennea-7045


In [19]:
resp = requests.get("https://worldathletics.org/results/diamond-league-meetings/2021/weltklasse-zurich-7058", headers=HEADERS, timeout=15)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

links = soup.find_all("a", href=True)
event_links = [a["href"] for a in links if "100-metres" in a["href"].lower()]

print(f"Found {len(event_links)} candidate 100m links")
for link in set(event_links):
    print(link)

Found 5 candidate 100m links
/results/diamond-league-meetings/2021/weltklasse-zurich-7058/men/100-metres/final/result#resultheader
/records/by-discipline/sprints/100-metres/all/men
/records/all-time-toplists/sprints/100-metres/outdoor/women/senior
/results/diamond-league-meetings/2021/weltklasse-zurich-7058/women/100-metres/final/result#resultheader
/results/diamond-league-meetings/2021/weltklasse-zurich-7058/women/100-metres-hurdles/final/result#resultheader


In [20]:
def get_mens_100m_url(meeting_url):
    """Given a Diamond League meeting page URL, find its men's 100m final result URL."""
    resp = requests.get(meeting_url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    
    soup = BeautifulSoup(resp.text, "html.parser")
    links = soup.find_all("a", href=True)
    
    event_links = [a["href"] for a in links if "/men/100-metres/final/" in a["href"]]
    
    if not event_links:
        return None  # this meeting had no men's 100m final (rare, but possible)
    
    href = event_links[0].split("#")[0]  # strip the #resultheader fragment
    return "https://worldathletics.org" + href if href.startswith("/") else href
# Test on one first
print(get_mens_100m_url("https://worldathletics.org/results/diamond-league-meetings/2021/weltklasse-zurich-7058"))

# Now find the men's 100m URL for every meeting
race_urls = []
for meeting_url in meeting_links:
    try:
        race_url = get_mens_100m_url(meeting_url)
        if race_url:
            race_urls.append(race_url)
            print(f"OK: {meeting_url} -> {race_url}")
        else:
            print(f"NO 100M EVENT: {meeting_url}")
    except Exception as e:
        print(f"FAILED: {meeting_url} — {e}")

print(f"\nCollected {len(race_urls)} race URLs total")

https://worldathletics.org/results/diamond-league-meetings/2021/weltklasse-zurich-7058/men/100-metres/final/result
OK: https://worldathletics.org/results/diamond-league-meetings/2021/weltklasse-zurich-7058 -> https://worldathletics.org/results/diamond-league-meetings/2021/weltklasse-zurich-7058/men/100-metres/final/result
OK: https://worldathletics.org/results/diamond-league-meetings/2021/allianz-memorial-van-damme-7056 -> https://worldathletics.org/results/diamond-league-meetings/2021/allianz-memorial-van-damme-7056/men/100-metres/final/result
OK: https://worldathletics.org/results/diamond-league-meetings/2021/meeting-de-paris-7055 -> https://worldathletics.org/results/diamond-league-meetings/2021/meeting-de-paris-7055/men/100-metres/final/result
NO 100M EVENT: https://worldathletics.org/results/diamond-league-meetings/2021/athletissima-lausanne-7054
OK: https://worldathletics.org/results/diamond-league-meetings/2021/prefontaine-classic-7044 -> https://worldathletics.org/results/diamo

In [21]:
resp = requests.get("https://worldathletics.org/results/diamond-league-meetings/2021/athletissima-lausanne-7054", headers=HEADERS, timeout=15)
soup = BeautifulSoup(resp.text, "html.parser")

# Print ALL links on the page, not just ones matching our filter
links = soup.find_all("a", href=True)
for a in links:
    href = a["href"]
    if "results/diamond-league-meetings" in href or "100-metres" in href.lower() or "200-metres" in href.lower():
        print(href)

/records/all-time-toplists/sprints/100-metres/outdoor/women/senior
/records/by-discipline/sprints/100-metres/all/men
/results/diamond-league-meetings
http://www.facebook.com/share.php?u=https://worldathletics.org/results/diamond-league-meetings/2021/athletissima-lausanne-7147613&title=Athletissima Lausanne
http://twitter.com/intent/tweet?status=Athletissima Lausanne+https://worldathletics.org/results/diamond-league-meetings/2021/athletissima-lausanne-7147613
/results/diamond-league-meetings/2021/athletissima-lausanne-7054/men/high-jump/final/series#resultheader
/results/diamond-league-meetings/2021/athletissima-lausanne-7054/men/200-metres/final/result#resultheader
/results/diamond-league-meetings/2021/athletissima-lausanne-7054/men/400-metres/final/result#resultheader
/results/diamond-league-meetings/2021/athletissima-lausanne-7054/men/800-metres/final/result#resultheader
/results/diamond-league-meetings/2021/athletissima-lausanne-7054/men/3000-metres/final/result#resultheader
/result

In [22]:
all_races = []
for u in race_urls:
    try:
        race = scrape_race(u)
        all_races.append(race)
        print(f"OK ({len(race)} rows): {u}")
    except Exception as e:
        print(f"FAILED: {u}")
        print(f"   error: {e}")

diamond_league_dataset = pd.concat(all_races, ignore_index=True)
print(f"\nTotal new rows: {len(diamond_league_dataset)}")
diamond_league_dataset.head()

OK (9 rows): https://worldathletics.org/results/diamond-league-meetings/2021/weltklasse-zurich-7058/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/results/diamond-league-meetings/2021/allianz-memorial-van-damme-7056/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/results/diamond-league-meetings/2021/meeting-de-paris-7055/men/100-metres/final/result
OK (9 rows): https://worldathletics.org/results/diamond-league-meetings/2021/prefontaine-classic-7044/men/100-metres/final/result
FAILED: https://worldathletics.org/results/diamond-league-meetings/2021/muller-british-gp-7051/men/100-metres/final/result
   error: Missing optional dependency 'html5lib'.  Use pip or conda to install html5lib.
OK (8 rows): https://worldathletics.org/results/diamond-league-meetings/2021/herculis-7052/men/100-metres/final/result
FAILED: https://worldathletics.org/results/diamond-league-meetings/2021/bauhaus-galan-7050/men/100-metres/final/result
   error: Missing optional d

C:\Users\rotem\AppData\Local\Temp\ipykernel_16424\4287459020.py:11: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  diamond_league_dataset = pd.concat(all_races, ignore_index=True)


,ATHLETE,COUNTRY,time,record_flag,Reaction Time,wind,date,venue,meet_name,source_url
0,Fred KERLEY,USA,9.87,NaN,NaN,-0.4,08/09/2021 00:00:00,None,World Athletics | 100 Metres Result | Weltkla...,https://worldathletics.org/results/diamond-lea...
1,Andre DE GRASSE,CAN,9.89,NaN,NaN,-0.4,08/09/2021 00:00:00,None,World Athletics | 100 Metres Result | Weltkla...,https://worldathletics.org/results/diamond-lea...
2,Ronnie BAKER,USA,9.91,NaN,NaN,-0.4,08/09/2021 00:00:00,None,World Athletics | 100 Metres Result | Weltkla...,https://worldathletics.org/results/diamond-lea...
3,Trayvon BROMELL,USA,9.96,NaN,NaN,-0.4,08/09/2021 00:00:00,None,World Athletics | 100 Metres Result | Weltkla...,https://worldathletics.org/results/diamond-lea...
4,Akani SIMBINE,RSA,10.10,NaN,NaN,-0.4,08/09/2021 00:00:00,None,World Athletics | 100 Metres Result | Weltkla...,https://worldathletics.org/results/diamond-lea...


In [23]:
!pip install html5lib

Defaulting to user installation because normal site-packages is not writeable
